# Download data files

To download data, use the same code as in the "Load an index file" notebook, except use 
`do_download=True` when calling the `get_files_between_dates` method.  Below is an 
example for downloading magnetometer data at 1 s resolution.

In [1]:
import numpy as np
import datetime as dt

from junopy import index
from junopy import config

# Set the junopy root directory to wherever you want the data to be downloaded.
config.ConfigVars.set_root(".junodata")   # <---- Modify as necessary.

# Define start and stop times.
start = dt.datetime(2020, 1, 4, 2)
stop = dt.datetime(2020, 1, 4, 23)

# Get the magnetometer index file.
mag_index = index.IndexFile(dataset='mag')

# Download files between start and stop.
mag_files = mag_index.get_files_between_dates(start, stop, sid="SS 1 SECOND", do_download=True)


Downloaded: /home/ysar/code/junopy/docs/examples/.junodata/data/mag/2020/004/fgm_jno_l3_2020004ss_r1s_v01.sts


In [2]:
mag_files

[PosixPath('/home/ysar/code/junopy/docs/examples/.junodata/data/mag/2020/004/fgm_jno_l3_2020004ss_r1s_v01.sts')]

Notice that the magnetometer files have the `.sts` extension.  Once you read them into a `MagData` object, these
files will be converted into `.hdf5`.

In [3]:
from junopy import mag

for mag_file in mag_files:
    # Reads and converts data into HDF5 automatically.
    _ = mag.MagData(filename=mag_file, is_hdf=False)

Calling `get_files_between_dates` again for the same time period will not re-download the data if the `.sts` 
or the converted `.hdf5` file is already present.

In [4]:
mag_files = mag_index.get_files_between_dates(start, stop, sid="SS 1 SECOND", do_download=True)
mag_files

[PosixPath('/home/ysar/code/junopy/docs/examples/.junodata/data/mag/2020/004/fgm_jno_l3_2020004ss_r1s_v01.sts')]

However, the returned list still contains the `.sts` extension.  So, when using the `get_files_between_dates` method to get 
a list of data files, keep in mind that the files may already exist in a different extension if they have already been 
read.  This is relevant for the magnetometer, Waves, and JEDI datasets.

You need to also know what `sid=` parameter to use for each instrument.  A description of different SID to use for downloading different types of data
is provided in `junopy.constants.SID` as a dictionary.

In [5]:
# Defined in junopy.constants, repeated here for reference.

# These are strings that are used to filter and identify specific data 
# in the PDS INDEX.TAB files.
SID = {
   'mag_1s': "SS 1 SECOND",      # For magnetometer data, junopy defaults to 
   'mag_64s': "SS 1 MINUTE",     # the Juno_JSS coordinate system. 
   'mag_native': "SUNSTATE",
   'waves_survey': "SURVEY",     # Identified both E and B spectra in survey mode.
   'jedi_electrons': "ERSESP",   # prefix 'HI' for high-res, 'LO' for low-res, or leave as is for both.
   'jedi_protons': "ERSTOFXER",  # prefix 'HI' for high-res, 'LO' for low-res, or leave as is for both.
   'jedi_heavies': "NONPTOFXPHR",
   'jedi_proton_pulseheight': "ERSTOFXPHR",
   'jedi_heavies_pulseheight': "NONPTOFXER",   # Verify this!! 
   'jade_electrons': "_ELC_",    # Gets both low-res and high-res electron data.
   'jade_species': 'ION_ANY',    # JADE ion species data.
   'jade_tof': 'ION_TOF',        # JADE ion time-of-flight data. 
} 